# 03 · Evaluación en la validación propia

Validación: 2 esquemas que **no** están en entrenamiento (`clinica_vet`, `inv_mgmt`) con 40 preguntas escritas por nosotros (14 fáciles, 14 medias, 12 difíciles). Métrica: EX ponderado (1/2/3) por ejecución con las reglas del enunciado.

Corridas:
1. **Baseline**: modelo base sin adapter (mismo prompt).
2. **LoRA A** y **LoRA B** sin autocorrección.
3. **Experimento de inferencia**: la mejor LoRA **con** autocorrección usando `ejecutar` (reintenta con el error de MySQL, máx. 3 llamadas).

Cada corrida va en un proceso aparte (libera la GPU) y se agrega a `resultados/validacion_propia.csv`.

In [ ]:
# Obtener el código: subir proyecto_3_deep.zip a /content (panel Archivos) o clonar el repo privado
import os, getpass
REPO = "proyecto_3_deep"
if not os.path.exists(f"/content/{REPO}/solucion.py"):
    if os.path.exists(f"/content/{REPO}.zip"):
        !unzip -q -o /content/{REPO}.zip -d /content/{REPO}
        # si el zip traía la carpeta raíz adentro, subirla un nivel
        if not os.path.exists(f"/content/{REPO}/solucion.py") and os.path.exists(f"/content/{REPO}/{REPO}/solucion.py"):
            !mv /content/{REPO}/{REPO}/* /content/{REPO}/ && rm -rf /content/{REPO}/{REPO}
    else:
        usuario = input("Usuario de GitHub: ")
        token = getpass.getpass("Token de GitHub: ")
        url_repo = input("URL del repo (https://github.com/<owner>/<repo>.git): ").replace("https://", f"https://{usuario}:{token}@")
        !git clone -q {url_repo} /content/{REPO}
%cd /content/{REPO}
!pip install -q -r requirements.txt

In [ ]:
# MySQL 8.4.10 con configuración por defecto (ONLY_FULL_GROUP_BY incluido)
!bash scripts/setup_mysql.sh
import os
os.environ["MYSQL_PORT"] = "3306"
os.environ["MYSQL_PASSWORD"] = "root"

In [ ]:
# Subir adapter_A.zip / adapter_B.zip si no están en el repo
import os
for c in ("A", "B"):
    if not os.path.exists(f"adapter_{c}") and os.path.exists(f"/content/adapter_{c}.zip"):
        !unzip -q -o /content/adapter_{c}.zip -d .
!ls -d adapter*

In [ ]:
!python scripts/evaluador.py --preparar
!python scripts/evaluador.py --oraculo | tail -1
!rm -f resultados/validacion_propia.csv resultados/resumen_experimentos.jsonl

In [ ]:
!python scripts/evaluador.py --nombre baseline --usar-adapter 0 --autocorreccion 0

In [ ]:
!python scripts/evaluador.py --nombre lora_A --adapter adapter_A/ --autocorreccion 0

In [ ]:
!python scripts/evaluador.py --nombre lora_B --adapter adapter_B/ --autocorreccion 0

In [ ]:
import pandas as pd
res = pd.read_json("resultados/resumen_experimentos.jsonl", lines=True)
MEJOR = "B" if res.set_index("config").loc["lora_B", "EX"] >= res.set_index("config").loc["lora_A", "EX"] else "A"
print("Mejor configuración:", MEJOR)

In [ ]:
# Experimento de inferencia: misma LoRA, ahora con autocorrección vía ejecutar()
!python scripts/evaluador.py --nombre lora_{MEJOR}_autocorr --adapter adapter_{MEJOR}/ --autocorreccion 1
# Baseline con autocorrección, para medir el aporte del adapter en el modo entregado
!python scripts/evaluador.py --nombre baseline_autocorr --usar-adapter 0 --autocorreccion 1

In [ ]:
res = pd.read_json("resultados/resumen_experimentos.jsonl", lines=True)
tabla = pd.concat([res[["config", "EX", "ejecutan", "tiempo_medio_s", "tiempo_max_s"]],
                   pd.json_normalize(res["por_nivel"])], axis=1)
tabla.to_csv("resultados/resumen_experimentos.csv", index=False)
display(tabla)

## Análisis de errores

In [ ]:
val = pd.read_csv("resultados/validacion_propia.csv")
errores = val[val.acierto == 0]
cat = errores.groupby(["config", "categoria"]).size().unstack(fill_value=0)
cat.to_csv("resultados/errores_por_categoria.csv")
display(cat)
display(errores[errores.config == f"lora_{MEJOR}_autocorr"][["id", "nivel", "categoria", "pregunta", "sql_pred", "error"]])

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
tabla.set_index("config")[["facil", "media", "dificil"]].plot.bar(ax=axs[0], rot=20)
axs[0].set_title("Exactitud por nivel"); axs[0].set_ylim(0, 1)
cat.plot.bar(stacked=True, ax=axs[1], rot=20)
axs[1].set_title("Errores por categoría")
fig.tight_layout(); fig.savefig("resultados/resultados_validacion.png", dpi=130)

In [ ]:
# Dejar el mejor adapter como entregable (adapter/) y bajar los resultados
!rm -rf adapter && cp -r adapter_{MEJOR} adapter
!zip -qr /content/resultados.zip resultados adapter
from google.colab import files
files.download("/content/resultados.zip")

## Verificación del contrato (como lo correrá el docente)
Carga < 5 min, < 30 s por pregunta, salida determinista, `usar_adapter` true/false.

In [ ]:
import time, json, importlib, sys
sys.path.insert(0, "scripts"); sys.path.insert(0, ".")
import mysql_utils as mu
t0 = time.time(); import solucion; print(f"carga: {time.time()-t0:.1f}s")
esq = mu.esquema_show_create("clinica_vet")
preg = "¿Cuántas consultas atendió cada veterinario? Devuelve nombre y número de consultas."
s1 = solucion.generar_sql(esq, preg, mu.hacer_ejecutar("clinica_vet"))
s2 = solucion.generar_sql(esq, preg, mu.hacer_ejecutar("clinica_vet"))
print(s1); print("determinista:", s1 == s2)